In [19]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().parent
sys.path.append(str(PROJECT_ROOT))

In [15]:
import pandas as pd
from src.ingestion.aemo import load_aemo_directory
df = load_aemo_directory()

In [3]:
df.shape

(240, 11)

In [16]:
df.head()

,RECORD_TYPE,DATASET,DATA_TYPE,VERSION,REGIONID,INTERVAL_DATETIME,OPERATIONAL_DEMAND,OPERATIONAL_DEMAND_ADJUSTMENT,WDR_ESTIMATE,LASTCHANGED,SOURCE_FILE
0,D,OPERATIONAL_DEMAND,ACTUAL,3,NSW1,2026/05/20 04:30:00,6656,0,0,2026/05/20 04:30:01,PUBLIC_ACTUAL_OPERATIONAL_DEMAND_DAILY_2026052...
1,D,OPERATIONAL_DEMAND,ACTUAL,3,NSW1,2026/05/20 05:00:00,6732,0,0,2026/05/20 05:00:02,PUBLIC_ACTUAL_OPERATIONAL_DEMAND_DAILY_2026052...
2,D,OPERATIONAL_DEMAND,ACTUAL,3,NSW1,2026/05/20 05:30:00,6977,0,0,2026/05/20 05:30:01,PUBLIC_ACTUAL_OPERATIONAL_DEMAND_DAILY_2026052...
3,D,OPERATIONAL_DEMAND,ACTUAL,3,NSW1,2026/05/20 06:00:00,7292,0,0,2026/05/20 06:00:01,PUBLIC_ACTUAL_OPERATIONAL_DEMAND_DAILY_2026052...
4,D,OPERATIONAL_DEMAND,ACTUAL,3,NSW1,2026/05/20 06:30:00,7830,0,0,2026/05/20 06:30:02,PUBLIC_ACTUAL_OPERATIONAL_DEMAND_DAILY_2026052...


In [ ]:
df.info()

In [ ]:
df["REGIONID"].value_counts()

In [17]:
df.isna().sum()

RECORD_TYPE                      0
DATASET                          0
DATA_TYPE                        0
VERSION                          0
REGIONID                         0
INTERVAL_DATETIME                0
OPERATIONAL_DEMAND               0
OPERATIONAL_DEMAND_ADJUSTMENT    0
WDR_ESTIMATE                     0
LASTCHANGED                      0
SOURCE_FILE                      0
dtype: int64

In [5]:
df.columns.tolist()

['RECORD_TYPE',
 'DATASET',
 'DATA_TYPE',
 'VERSION',
 'REGIONID',
 'INTERVAL_DATETIME',
 'OPERATIONAL_DEMAND',
 'OPERATIONAL_DEMAND_ADJUSTMENT',
 'WDR_ESTIMATE',
 'LASTCHANGED',
 'SOURCE_FILE']

In [18]:
import importlib
import src.processing.clean_aemo as clean_aemo_module

importlib.reload(clean_aemo_module)

from src.processing.clean_aemo import (
    clean_operational_demand,
    validate_operational_demand,
)


In [19]:
vic_df = clean_operational_demand(df, region="VIC1")
report = validate_operational_demand(vic_df)

In [20]:
vic_df.head()

,RECORD_TYPE,DATASET,DATA_TYPE,VERSION,REGIONID,INTERVAL_DATETIME,OPERATIONAL_DEMAND,OPERATIONAL_DEMAND_ADJUSTMENT,WDR_ESTIMATE,LASTCHANGED,SOURCE_FILE
0,D,OPERATIONAL_DEMAND,ACTUAL,3,VIC1,2026-05-20 04:30:00,4726,0,0,2026-05-20 04:30:01,PUBLIC_ACTUAL_OPERATIONAL_DEMAND_DAILY_2026052...
1,D,OPERATIONAL_DEMAND,ACTUAL,3,VIC1,2026-05-20 05:00:00,4757,0,0,2026-05-20 05:00:02,PUBLIC_ACTUAL_OPERATIONAL_DEMAND_DAILY_2026052...
2,D,OPERATIONAL_DEMAND,ACTUAL,3,VIC1,2026-05-20 05:30:00,4943,0,0,2026-05-20 05:30:01,PUBLIC_ACTUAL_OPERATIONAL_DEMAND_DAILY_2026052...
3,D,OPERATIONAL_DEMAND,ACTUAL,3,VIC1,2026-05-20 06:00:00,5158,0,0,2026-05-20 06:00:01,PUBLIC_ACTUAL_OPERATIONAL_DEMAND_DAILY_2026052...
4,D,OPERATIONAL_DEMAND,ACTUAL,3,VIC1,2026-05-20 06:30:00,5571,0,0,2026-05-20 06:30:02,PUBLIC_ACTUAL_OPERATIONAL_DEMAND_DAILY_2026052...


In [21]:
vic_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 48 entries, 0 to 47
Data columns (total 11 columns):
 #   Column                         Non-Null Count  Dtype         
---  ------                         --------------  -----         
 0   RECORD_TYPE                    48 non-null     str           
 1   DATASET                        48 non-null     str           
 2   DATA_TYPE                      48 non-null     str           
 3   VERSION                        48 non-null     str           
 4   REGIONID                       48 non-null     str           
 5   INTERVAL_DATETIME              48 non-null     datetime64[us]
 6   OPERATIONAL_DEMAND             48 non-null     int64         
 7   OPERATIONAL_DEMAND_ADJUSTMENT  48 non-null     int64         
 8   WDR_ESTIMATE                   48 non-null     int64         
 9   LASTCHANGED                    48 non-null     datetime64[us]
 10  SOURCE_FILE                    48 non-null     str           
dtypes: datetime64[us](2), int64(3), 

In [22]:
report

{'rows': 48,
 'start_datetime': Timestamp('2026-05-20 04:30:00'),
 'end_datetime': Timestamp('2026-05-21 04:00:00'),
 'duplicate_intervals': 0,
 'missing_interval_count': 0,
 'missing_intervals': [],
 'missing_values': {'RECORD_TYPE': 0,
  'DATASET': 0,
  'DATA_TYPE': 0,
  'VERSION': 0,
  'REGIONID': 0,
  'INTERVAL_DATETIME': 0,
  'OPERATIONAL_DEMAND': 0,
  'OPERATIONAL_DEMAND_ADJUSTMENT': 0,
  'WDR_ESTIMATE': 0,
  'LASTCHANGED': 0,
  'SOURCE_FILE': 0}}

In [25]:
import importlib
import src.processing.prepare_model_data as prepare_module

importlib.reload(prepare_module)

from src.processing.prepare_model_data import prepare_model_data


In [26]:
model_df = prepare_model_data(vic_df)

model_df.head()

,INTERVAL_DATETIME,OPERATIONAL_DEMAND
0,2026-05-20 04:30:00,4726
1,2026-05-20 05:00:00,4757
2,2026-05-20 05:30:00,4943
3,2026-05-20 06:00:00,5158
4,2026-05-20 06:30:00,5571


In [27]:
import importlib
import src.features.build_calendar_features as calendar_module

importlib.reload(calendar_module)

from src.features.build_calendar_features import add_calendar_features

feature_df = add_calendar_features(model_df)

feature_df.head()

,INTERVAL_DATETIME,OPERATIONAL_DEMAND,year,month,day,day_of_week,hour,minute,is_weekend
0,2026-05-20 04:30:00,4726,2026,5,20,2,4,30,0
1,2026-05-20 05:00:00,4757,2026,5,20,2,5,0,0
2,2026-05-20 05:30:00,4943,2026,5,20,2,5,30,0
3,2026-05-20 06:00:00,5158,2026,5,20,2,6,0,0
4,2026-05-20 06:30:00,5571,2026,5,20,2,6,30,0


In [28]:
import importlib
import src.features.build_lag_features as lag_module

importlib.reload(lag_module)

from src.features.build_lag_features import add_lag_features

lag_df = add_lag_features(feature_df)

lag_df.head(5)

,INTERVAL_DATETIME,OPERATIONAL_DEMAND,year,month,day,day_of_week,hour,minute,is_weekend,lag_1,lag_2,lag_48,lag_336
0,2026-05-20 04:30:00,4726,2026,5,20,2,4,30,0,NaN,NaN,NaN,NaN
1,2026-05-20 05:00:00,4757,2026,5,20,2,5,0,0,4726.0,NaN,NaN,NaN
2,2026-05-20 05:30:00,4943,2026,5,20,2,5,30,0,4757.0,4726.0,NaN,NaN
3,2026-05-20 06:00:00,5158,2026,5,20,2,6,0,0,4943.0,4757.0,NaN,NaN
4,2026-05-20 06:30:00,5571,2026,5,20,2,6,30,0,5158.0,4943.0,NaN,NaN


In [29]:
import importlib
import src.features.build_rolling_features as rolling_module

importlib.reload(rolling_module)

from src.features.build_rolling_features import add_rolling_features

rolling_df = add_rolling_features(lag_df)

rolling_df.head(10)

,INTERVAL_DATETIME,OPERATIONAL_DEMAND,year,month,day,day_of_week,hour,minute,is_weekend,lag_1,...,lag_48,lag_336,rolling_mean_4,rolling_std_4,rolling_min_4,rolling_max_4,rolling_mean_48,rolling_std_48,rolling_min_48,rolling_max_48
0,2026-05-20 04:30:00,4726,2026,5,20,2,4,30,0,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2026-05-20 05:00:00,4757,2026,5,20,2,5,0,0,4726.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2026-05-20 05:30:00,4943,2026,5,20,2,5,30,0,4757.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,2026-05-20 06:00:00,5158,2026,5,20,2,6,0,0,4943.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,2026-05-20 06:30:00,5571,2026,5,20,2,6,30,0,5158.0,...,NaN,NaN,4896.00,199.226839,4726.0,5158.0,NaN,NaN,NaN,NaN
5,2026-05-20 07:00:00,6033,2026,5,20,2,7,0,0,5571.0,...,NaN,NaN,5107.25,349.901296,4757.0,5571.0,NaN,NaN,NaN,NaN
6,2026-05-20 07:30:00,6470,2026,5,20,2,7,30,0,6033.0,...,NaN,NaN,5426.25,481.174518,4943.0,6033.0,NaN,NaN,NaN,NaN
7,2026-05-20 08:00:00,6760,2026,5,20,2,8,0,0,6470.0,...,NaN,NaN,5808.00,567.901987,5158.0,6470.0,NaN,NaN,NaN,NaN
8,2026-05-20 08:30:00,6801,2026,5,20,2,8,30,0,6760.0,...,NaN,NaN,6208.50,519.532161,5571.0,6760.0,NaN,NaN,NaN,NaN
9,2026-05-20 09:00:00,6606,2026,5,20,2,9,0,0,6801.0,...,NaN,NaN,6516.00,354.102622,6033.0,6801.0,NaN,NaN,NaN,NaN


In [30]:
import importlib
import src.features.build_cyclical_features as cyclical_module

importlib.reload(cyclical_module)

from src.features.build_cyclical_features import add_cyclical_features

cyclical_df = add_cyclical_features(rolling_df)

cyclical_df.head()

,INTERVAL_DATETIME,OPERATIONAL_DEMAND,year,month,day,day_of_week,hour,minute,is_weekend,lag_1,...,rolling_std_48,rolling_min_48,rolling_max_48,time_of_day,time_sin,time_cos,day_of_week_sin,day_of_week_cos,month_sin,month_cos
0,2026-05-20 04:30:00,4726,2026,5,20,2,4,30,0,NaN,...,NaN,NaN,NaN,4.5,0.923880,3.826834e-01,0.974928,-0.222521,0.866025,-0.5
1,2026-05-20 05:00:00,4757,2026,5,20,2,5,0,0,4726.0,...,NaN,NaN,NaN,5.0,0.965926,2.588190e-01,0.974928,-0.222521,0.866025,-0.5
2,2026-05-20 05:30:00,4943,2026,5,20,2,5,30,0,4757.0,...,NaN,NaN,NaN,5.5,0.991445,1.305262e-01,0.974928,-0.222521,0.866025,-0.5
3,2026-05-20 06:00:00,5158,2026,5,20,2,6,0,0,4943.0,...,NaN,NaN,NaN,6.0,1.000000,6.123234e-17,0.974928,-0.222521,0.866025,-0.5
4,2026-05-20 06:30:00,5571,2026,5,20,2,6,30,0,5158.0,...,NaN,NaN,NaN,6.5,0.991445,-1.305262e-01,0.974928,-0.222521,0.866025,-0.5


In [4]:
import importlib

import src.ingestion.aemo as aemo_module
import src.ingestion.load_historical_operational_demand as historical_module

importlib.reload(aemo_module)
importlib.reload(historical_module)

from src.ingestion.load_historical_operational_demand import (
    load_historical_operational_demand,
)
from src.processing.clean_aemo import clean_operational_demand
from src.processing.prepare_model_data import prepare_model_data

In [5]:
archive_directory = (
    PROJECT_ROOT
    / "data"
    / "raw"
    / "aemo"
    / "operational_demand"
    / "archive"
)

historical_raw_df = load_historical_operational_demand(
    archive_directory
)

Reading PUBLIC_ACTUAL_OPERATIONAL_DEMAND_DAILY_20250601.zip
Reading PUBLIC_ACTUAL_OPERATIONAL_DEMAND_DAILY_20250701.zip
Reading PUBLIC_ACTUAL_OPERATIONAL_DEMAND_DAILY_20250801.zip
Reading PUBLIC_ACTUAL_OPERATIONAL_DEMAND_DAILY_20250901.zip
Reading PUBLIC_ACTUAL_OPERATIONAL_DEMAND_DAILY_20251001.zip
Reading PUBLIC_ACTUAL_OPERATIONAL_DEMAND_DAILY_20251101.zip
Reading PUBLIC_ACTUAL_OPERATIONAL_DEMAND_DAILY_20251201.zip
Reading PUBLIC_ACTUAL_OPERATIONAL_DEMAND_DAILY_20260101.zip
Reading PUBLIC_ACTUAL_OPERATIONAL_DEMAND_DAILY_20260201.zip
Reading PUBLIC_ACTUAL_OPERATIONAL_DEMAND_DAILY_20260301.zip
Reading PUBLIC_ACTUAL_OPERATIONAL_DEMAND_DAILY_20260401.zip
Reading PUBLIC_ACTUAL_OPERATIONAL_DEMAND_DAILY_20260501.zip

Loaded 12 monthly archives, 365 daily archives, 365 CSV files and 87,600 data rows.


In [6]:
historical_raw_df.shape

(87600, 11)

In [7]:
historical_raw_df.columns.tolist()

['RECORD_TYPE',
 'DATASET',
 'DATA_TYPE',
 'VERSION',
 'REGIONID',
 'INTERVAL_DATETIME',
 'OPERATIONAL_DEMAND',
 'OPERATIONAL_DEMAND_ADJUSTMENT',
 'WDR_ESTIMATE',
 'LASTCHANGED',
 'SOURCE_FILE']

In [8]:
historical_raw_df["REGIONID"].value_counts()

REGIONID
NSW1    17520
QLD1    17520
SA1     17520
TAS1    17520
VIC1    17520
Name: count, dtype: int64

In [10]:
historical_clean_df = clean_operational_demand(
    historical_raw_df,
    region="VIC1",
)

In [11]:
from src.processing.clean_aemo import validate_operational_demand

validation_result = validate_operational_demand(
    historical_clean_df
)

validation_result

{'rows': 17520,
 'start_datetime': Timestamp('2025-06-01 04:30:00'),
 'end_datetime': Timestamp('2026-06-01 04:00:00'),
 'duplicate_intervals': 0,
 'missing_interval_count': 0,
 'missing_intervals': [],
 'missing_values': {'RECORD_TYPE': 0,
  'DATASET': 0,
  'DATA_TYPE': 0,
  'VERSION': 0,
  'REGIONID': 0,
  'INTERVAL_DATETIME': 0,
  'OPERATIONAL_DEMAND': 0,
  'OPERATIONAL_DEMAND_ADJUSTMENT': 0,
  'WDR_ESTIMATE': 0,
  'LASTCHANGED': 0,
  'SOURCE_FILE': 0}}

In [12]:
historical_model_df = prepare_model_data(
    historical_clean_df
)

historical_model_df.head()

,INTERVAL_DATETIME,OPERATIONAL_DEMAND
0,2025-06-01 04:30:00,4535
1,2025-06-01 05:00:00,4554
2,2025-06-01 05:30:00,4626
3,2025-06-01 06:00:00,4710
4,2025-06-01 06:30:00,4873


In [13]:
print(
    historical_model_df["INTERVAL_DATETIME"].min()
)

print(
    historical_model_df["INTERVAL_DATETIME"].max()
)

print(
    historical_model_df.shape
)

2025-06-01 04:30:00
2026-06-01 04:00:00
(17520, 2)


In [15]:
import importlib

import src.features.build_calendar_features as calendar_module
import src.features.build_lag_features as lag_module
import src.features.build_rolling_features as rolling_module
import src.features.build_cyclical_features as cyclical_module

importlib.reload(calendar_module)
importlib.reload(lag_module)
importlib.reload(rolling_module)
importlib.reload(cyclical_module)

from src.features.build_calendar_features import add_calendar_features
from src.features.build_lag_features import add_lag_features
from src.features.build_rolling_features import add_rolling_features
from src.features.build_cyclical_features import add_cyclical_features

In [16]:
historical_calendar_df = add_calendar_features(
    historical_model_df
)

historical_lag_df = add_lag_features(
    historical_calendar_df
)

historical_rolling_df = add_rolling_features(
    historical_lag_df
)

historical_feature_df = add_cyclical_features(
    historical_rolling_df
)

In [17]:
historical_feature_df 

,INTERVAL_DATETIME,OPERATIONAL_DEMAND,year,month,day,day_of_week,hour,minute,is_weekend,lag_1,...,rolling_std_48,rolling_min_48,rolling_max_48,time_of_day,time_sin,time_cos,day_of_week_sin,day_of_week_cos,month_sin,month_cos
0,2025-06-01 04:30:00,4535,2025,6,1,6,4,30,1,NaN,...,NaN,NaN,NaN,4.5,0.923880,3.826834e-01,-0.781831,0.62349,0.5,-0.866025
1,2025-06-01 05:00:00,4554,2025,6,1,6,5,0,1,4535.0,...,NaN,NaN,NaN,5.0,0.965926,2.588190e-01,-0.781831,0.62349,0.5,-0.866025
2,2025-06-01 05:30:00,4626,2025,6,1,6,5,30,1,4554.0,...,NaN,NaN,NaN,5.5,0.991445,1.305262e-01,-0.781831,0.62349,0.5,-0.866025
3,2025-06-01 06:00:00,4710,2025,6,1,6,6,0,1,4626.0,...,NaN,NaN,NaN,6.0,1.000000,6.123234e-17,-0.781831,0.62349,0.5,-0.866025
4,2025-06-01 06:30:00,4873,2025,6,1,6,6,30,1,4710.0,...,NaN,NaN,NaN,6.5,0.991445,-1.305262e-01,-0.781831,0.62349,0.5,-0.866025
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
17515,2026-06-01 02:00:00,4916,2026,6,1,0,2,0,0,5063.0,...,796.781644,4578.0,7120.0,2.0,0.500000,8.660254e-01,0.000000,1.00000,0.5,-0.866025
17516,2026-06-01 02:30:00,4808,2026,6,1,0,2,30,0,4916.0,...,797.264392,4578.0,7120.0,2.5,0.608761,7.933533e-01,0.000000,1.00000,0.5,-0.866025
17517,2026-06-01 03:00:00,4728,2026,6,1,0,3,0,0,4808.0,...,797.351649,4578.0,7120.0,3.0,0.707107,7.071068e-01,0.000000,1.00000,0.5,-0.866025
17518,2026-06-01 03:30:00,4714,2026,6,1,0,3,30,0,4728.0,...,796.993445,4578.0,7120.0,3.5,0.793353,6.087614e-01,0.000000,1.00000,0.5,-0.866025


In [18]:
historical_feature_df.isna().sum()

INTERVAL_DATETIME       0
OPERATIONAL_DEMAND      0
year                    0
month                   0
day                     0
day_of_week             0
hour                    0
minute                  0
is_weekend              0
lag_1                   1
lag_2                   2
lag_48                 48
lag_336               336
rolling_mean_4          4
rolling_std_4           4
rolling_min_4           4
rolling_max_4           4
rolling_mean_48        48
rolling_std_48         48
rolling_min_48         48
rolling_max_48         48
time_of_day             0
time_sin                0
time_cos                0
day_of_week_sin         0
day_of_week_cos         0
month_sin               0
month_cos               0
dtype: int64

In [ ]:
historical_feature_df = (
    historical_feature_df
    .dropna()
    .reset_index(drop=True)
)

In [21]:
import importlib
import src.processing.prepare_training_data as training_module

importlib.reload(training_module)

from src.processing.prepare_training_data import prepare_training_data

In [25]:
# Notebook import

import importlib
import src.ingestion.weather as weather_module

importlib.reload(weather_module)

from src.ingestion.weather import load_historical_weather

In [32]:
# Re-download the weather data using fixed AEST.

# AEMO/NEM timestamps use Australian Eastern Standard Time:
# UTC+10 throughout the year, without daylight-saving changes.
#
# Australia/Brisbane provides fixed UTC+10 while keeping the
# weather coordinates located in Melbourne.

weather_raw_df = load_historical_weather(
    start_date="2025-06-01",
    end_date="2026-06-01",
    latitude=-37.8136,
    longitude=144.9631,
    timezone="Australia/Brisbane",
)

print(weather_raw_df.attrs["timezone"])
print(weather_raw_df["time"].min())
print(weather_raw_df["time"].max())

Australia/Brisbane
2025-06-01T00:00
2026-06-01T23:00


In [33]:
print(weather_raw_df.shape)
print(weather_raw_df.head())
print(weather_raw_df.tail())



(8784, 8)
               time  temperature_2m  relative_humidity_2m  \
0  2025-06-01T00:00             7.3                    92   
1  2025-06-01T01:00             6.8                    91   
2  2025-06-01T02:00             6.6                    90   
3  2025-06-01T03:00             7.0                    88   
4  2025-06-01T04:00             7.4                    86   

   apparent_temperature  precipitation  cloud_cover  wind_speed_10m  \
0                   5.5            0.0           68             5.0   
1                   4.7            0.0          100             6.5   
2                   4.3            0.0          100             7.3   
3                   4.4            0.0           88             9.0   
4                   4.6            0.0          100            11.1   

   shortwave_radiation_instant  
0                          0.0  
1                          0.0  
2                          0.0  
3                          0.0  
4                          0.0 

In [29]:
print(weather_raw_df.columns.tolist())

['time', 'temperature_2m', 'relative_humidity_2m', 'apparent_temperature', 'precipitation', 'cloud_cover', 'wind_speed_10m', 'shortwave_radiation_instant']


In [34]:
print("Timezone:", weather_raw_df.attrs["timezone"])
print("UTC offset:", weather_raw_df.attrs["utc_offset_seconds"])
print("Units:", weather_raw_df.attrs["hourly_units"])


Timezone: Australia/Brisbane
UTC offset: 36000
Units: {'time': 'iso8601', 'temperature_2m': '°C', 'relative_humidity_2m': '%', 'apparent_temperature': '°C', 'precipitation': 'mm', 'cloud_cover': '%', 'wind_speed_10m': 'km/h', 'shortwave_radiation_instant': 'W/m²'}


In [35]:
print(weather_raw_df["time"].min())
print(weather_raw_df["time"].max())

print(
    "Duplicate timestamps:",
    weather_raw_df["time"].duplicated().sum(),
)

print("\nMissing values:")
print(weather_raw_df.isna().sum())

2025-06-01T00:00
2026-06-01T23:00
Duplicate timestamps: 0

Missing values:
time                           0
temperature_2m                 0
relative_humidity_2m           0
apparent_temperature           0
precipitation                  0
cloud_cover                    0
wind_speed_10m                 0
shortwave_radiation_instant    0
dtype: int64


In [36]:
# Notebook import

import importlib
import src.processing.clean_weather as clean_weather_module

importlib.reload(clean_weather_module)

from src.processing.clean_weather import clean_weather_data

In [37]:
weather_clean_df = clean_weather_data(
    weather_raw_df
)

In [39]:
print("Shape:", weather_clean_df.shape)

print(
    "\nDate range:",
    weather_clean_df["INTERVAL_DATETIME"].min(),
    "to",
    weather_clean_df["INTERVAL_DATETIME"].max(),
)

print(
    "\nDuplicate timestamps:",
    weather_clean_df["INTERVAL_DATETIME"]
    .duplicated()
    .sum(),
)

print("\nMissing values:")
print(weather_clean_df.isna().sum())

print("\nColumns:")
print(weather_clean_df.columns.tolist())

Shape: (17567, 9)

Date range: 2025-06-01 00:00:00 to 2026-06-01 23:00:00

Duplicate timestamps: 0

Missing values:
INTERVAL_DATETIME       0
temperature             0
relative_humidity       0
apparent_temperature    0
cloud_cover             0
wind_speed              0
solar_radiation         0
precipitation           0
is_raining              0
dtype: int64

Columns:
['INTERVAL_DATETIME', 'temperature', 'relative_humidity', 'apparent_temperature', 'cloud_cover', 'wind_speed', 'solar_radiation', 'precipitation', 'is_raining']


In [40]:
weather_clean_df.head(10)

,INTERVAL_DATETIME,temperature,relative_humidity,apparent_temperature,cloud_cover,wind_speed,solar_radiation,precipitation,is_raining
0,2025-06-01 00:00:00,7.30,92.0,5.50,68.0,5.00,0.0,0.0,0
1,2025-06-01 00:30:00,7.05,91.5,5.10,84.0,5.75,0.0,0.0,0
2,2025-06-01 01:00:00,6.80,91.0,4.70,100.0,6.50,0.0,0.0,0
3,2025-06-01 01:30:00,6.70,90.5,4.50,100.0,6.90,0.0,0.0,0
4,2025-06-01 02:00:00,6.60,90.0,4.30,100.0,7.30,0.0,0.0,0
5,2025-06-01 02:30:00,6.80,89.0,4.35,94.0,8.15,0.0,0.0,0
6,2025-06-01 03:00:00,7.00,88.0,4.40,88.0,9.00,0.0,0.0,0
7,2025-06-01 03:30:00,7.20,87.0,4.50,94.0,10.05,0.0,0.0,0
8,2025-06-01 04:00:00,7.40,86.0,4.60,100.0,11.10,0.0,0.0,0
9,2025-06-01 04:30:00,7.70,84.5,4.85,100.0,11.60,0.0,0.0,0


In [41]:
print(
    weather_clean_df[
        [
            "INTERVAL_DATETIME",
            "temperature",
            "precipitation",
            "is_raining",
        ]
    ].head(10)
)

    INTERVAL_DATETIME  temperature  precipitation  is_raining
0 2025-06-01 00:00:00         7.30            0.0           0
1 2025-06-01 00:30:00         7.05            0.0           0
2 2025-06-01 01:00:00         6.80            0.0           0
3 2025-06-01 01:30:00         6.70            0.0           0
4 2025-06-01 02:00:00         6.60            0.0           0
5 2025-06-01 02:30:00         6.80            0.0           0
6 2025-06-01 03:00:00         7.00            0.0           0
7 2025-06-01 03:30:00         7.20            0.0           0
8 2025-06-01 04:00:00         7.40            0.0           0
9 2025-06-01 04:30:00         7.70            0.0           0


In [42]:
print("Shape:", weather_clean_df.shape)

print(
    "Date range:",
    weather_clean_df["INTERVAL_DATETIME"].min(),
    "to",
    weather_clean_df["INTERVAL_DATETIME"].max(),
)

print(
    "Duplicate timestamps:",
    weather_clean_df["INTERVAL_DATETIME"]
    .duplicated()
    .sum(),
)

print("\nMissing values:")
print(weather_clean_df.isna().sum())

print("\nInterval counts:")
print(
    weather_clean_df["INTERVAL_DATETIME"]
    .diff()
    .value_counts()
    .head()
)

Shape: (17567, 9)
Date range: 2025-06-01 00:00:00 to 2026-06-01 23:00:00
Duplicate timestamps: 0

Missing values:
INTERVAL_DATETIME       0
temperature             0
relative_humidity       0
apparent_temperature    0
cloud_cover             0
wind_speed              0
solar_radiation         0
precipitation           0
is_raining              0
dtype: int64

Interval counts:
INTERVAL_DATETIME
0 days 00:30:00    17566
Name: count, dtype: int64


In [43]:
# Notebook import

import importlib
import src.processing.merge_demand_weather as merge_weather_module

importlib.reload(merge_weather_module)

from src.processing.merge_demand_weather import merge_demand_weather

In [44]:
merged_feature_df = merge_demand_weather(
    historical_feature_df,
    weather_clean_df,
)

In [45]:
print("Demand rows:", len(historical_feature_df))
print("Merged rows:", len(merged_feature_df))

print(
    "Date range:",
    merged_feature_df["INTERVAL_DATETIME"].min(),
    "to",
    merged_feature_df["INTERVAL_DATETIME"].max(),
)

print(
    "Duplicate timestamps:",
    merged_feature_df["INTERVAL_DATETIME"]
    .duplicated()
    .sum(),
)

print("\nMissing weather values:")
print(
    merged_feature_df[
        [
            "temperature",
            "relative_humidity",
            "apparent_temperature",
            "cloud_cover",
            "wind_speed",
            "solar_radiation",
            "precipitation",
            "is_raining",
        ]
    ].isna().sum()
)

Demand rows: 17520
Merged rows: 17520
Date range: 2025-06-01 04:30:00 to 2026-06-01 04:00:00
Duplicate timestamps: 0

Missing weather values:
temperature             0
relative_humidity       0
apparent_temperature    0
cloud_cover             0
wind_speed              0
solar_radiation         0
precipitation           0
is_raining              0
dtype: int64


In [46]:
merged_feature_df[
    [
        "INTERVAL_DATETIME",
        "OPERATIONAL_DEMAND",
        "temperature",
        "apparent_temperature",
        "relative_humidity",
        "solar_radiation",
        "precipitation",
        "is_raining",
    ]
].head(10)

,INTERVAL_DATETIME,OPERATIONAL_DEMAND,temperature,apparent_temperature,relative_humidity,solar_radiation,precipitation,is_raining
0,2025-06-01 04:30:00,4535,7.70,4.85,84.5,0.00,0.0,0
1,2025-06-01 05:00:00,4554,8.00,5.10,83.0,0.00,0.0,0
2,2025-06-01 05:30:00,4626,8.20,5.20,82.0,0.00,0.0,0
3,2025-06-01 06:00:00,4710,8.40,5.30,81.0,0.00,0.0,0
4,2025-06-01 06:30:00,4873,8.50,5.35,80.0,0.00,0.0,0
5,2025-06-01 07:00:00,5095,8.60,5.40,79.0,0.00,0.0,0
6,2025-06-01 07:30:00,5342,8.75,5.50,77.5,24.75,0.0,0
7,2025-06-01 08:00:00,5597,8.90,5.60,76.0,49.50,0.0,0
8,2025-06-01 08:30:00,5691,9.75,6.35,72.0,120.65,0.0,0
9,2025-06-01 09:00:00,5492,10.60,7.10,68.0,191.80,0.0,0


In [47]:
# Notebook import

import importlib
import src.features.build_weather_features as weather_features_module

importlib.reload(weather_features_module)

from src.features.build_weather_features import build_weather_features

In [48]:
weather_feature_df = build_weather_features(
    weather_clean_df,
    heating_base_temperature=18.0,
    cooling_base_temperature=22.0,
    high_temperature_threshold=30.0,
    high_humidity_threshold=80.0,
)

In [49]:
print("Before:", weather_clean_df.shape)
print("After :", weather_feature_df.shape)

print("\nNew feature columns:")
print(
    [
        column
        for column in weather_feature_df.columns
        if column not in weather_clean_df.columns
    ]
)

print("\nMissing values:")
print(
    weather_feature_df.isna().sum()[
        weather_feature_df.isna().sum() > 0
    ]
)

Before: (17567, 9)
After : (17567, 22)

New feature columns:
['heating_degree', 'cooling_degree', 'temperature_squared', 'apparent_temperature_difference', 'temperature_change_1h', 'temperature_change_3h', 'temperature_humidity_interaction', 'is_high_temperature', 'is_high_humidity', 'is_hot_and_humid', 'is_daylight', 'is_moderate_rain', 'is_heavy_rain']

Missing values:
temperature_change_1h    2
temperature_change_3h    6
dtype: int64


In [50]:
weather_feature_df[
    [
        "INTERVAL_DATETIME",
        "temperature",
        "heating_degree",
        "cooling_degree",
        "temperature_change_1h",
        "temperature_change_3h",
        "is_high_temperature",
        "is_hot_and_humid",
        "is_daylight",
        "is_raining",
    ]
].head(10)

,INTERVAL_DATETIME,temperature,heating_degree,cooling_degree,temperature_change_1h,temperature_change_3h,is_high_temperature,is_hot_and_humid,is_daylight,is_raining
0,2025-06-01 00:00:00,7.30,10.70,0.0,NaN,NaN,0,0,0,0
1,2025-06-01 00:30:00,7.05,10.95,0.0,NaN,NaN,0,0,0,0
2,2025-06-01 01:00:00,6.80,11.20,0.0,-0.50,NaN,0,0,0,0
3,2025-06-01 01:30:00,6.70,11.30,0.0,-0.35,NaN,0,0,0,0
4,2025-06-01 02:00:00,6.60,11.40,0.0,-0.20,NaN,0,0,0,0
5,2025-06-01 02:30:00,6.80,11.20,0.0,0.10,NaN,0,0,0,0
6,2025-06-01 03:00:00,7.00,11.00,0.0,0.40,-0.30,0,0,0,0
7,2025-06-01 03:30:00,7.20,10.80,0.0,0.40,0.15,0,0,0,0
8,2025-06-01 04:00:00,7.40,10.60,0.0,0.40,0.60,0,0,0,0
9,2025-06-01 04:30:00,7.70,10.30,0.0,0.50,1.00,0,0,0,0


In [51]:
# Merge the refined weather dataset with demand features.

merged_feature_df = merge_demand_weather(
    historical_feature_df,
    weather_feature_df,
)

In [52]:
WEATHER_COLUMNS = [
    "temperature",
    "relative_humidity",
    "apparent_temperature",
    "cloud_cover",
    "wind_speed",
    "solar_radiation",
    "precipitation",
    "is_raining",
    "heating_degree",
    "cooling_degree",
    "temperature_squared",
    "apparent_temperature_difference",
    "temperature_change_1h",
    "temperature_change_3h",
    "temperature_humidity_interaction",
    "is_high_temperature",
    "is_high_humidity",
    "is_hot_and_humid",
    "is_daylight",
    "is_moderate_rain",
    "is_heavy_rain",
]

In [53]:
# Reload the updated merge module.

importlib.reload(merge_weather_module)

from src.processing.merge_demand_weather import merge_demand_weather

merged_feature_df = merge_demand_weather(
    historical_feature_df,
    weather_feature_df,
)

In [54]:
print("Merged shape:", merged_feature_df.shape)

print(
    "Duplicate timestamps:",
    merged_feature_df["INTERVAL_DATETIME"]
    .duplicated()
    .sum(),
)

print("\nMissing values:")
print(
    merged_feature_df.isna().sum()[
        merged_feature_df.isna().sum() > 0
    ]
)

Merged shape: (17520, 36)
Duplicate timestamps: 0

Missing values:
lag_1                1
lag_2                2
lag_48              48
lag_336            336
rolling_mean_4       4
rolling_std_4        4
rolling_min_4        4
rolling_max_4        4
rolling_mean_48     48
rolling_std_48      48
rolling_min_48      48
rolling_max_48      48
dtype: int64


In [55]:
# Notebook import

import importlib
import src.processing.prepare_training_data as prepare_training_module

importlib.reload(prepare_training_module)

from src.processing.prepare_training_data import prepare_training_data

In [56]:
# Prepare final modelling dataset

training_df = prepare_training_data(
    merged_feature_df
)

In [57]:
# Validation

print("Before:", len(merged_feature_df))
print("After :", len(training_df))

print(
    "\nDate range:",
    training_df["INTERVAL_DATETIME"].min(),
    "to",
    training_df["INTERVAL_DATETIME"].max(),
)

print(
    "\nDuplicate timestamps:",
    training_df["INTERVAL_DATETIME"].duplicated().sum(),
)

print("\nMissing values:")
print(
    training_df.isna().sum()[
        training_df.isna().sum() > 0
    ]
)

print("\nShape:", training_df.shape)

Before: 17520
After : 17184

Date range: 2025-06-08 04:30:00 to 2026-06-01 04:00:00

Duplicate timestamps: 0

Missing values:
Series([], dtype: int64)

Shape: (17184, 36)


0     0.00
1     0.00
2     0.00
3     0.00
4     0.00
5     0.00
6     4.85
7     9.70
8    19.90
9    30.10
Name: solar_radiation, dtype: float64